Ollama is now installed. Ollama is an open-source tool that simplifies running large language models locally.
It provides an easy way to download, run, and manage various LLMs on your machine.

If you encounter any GPU-related issues, try the following troubleshooting steps:
1. Ensure CUDA is properly installed and configured.
2. Check if the correct GPU drivers are installed.
3. Verify that Ollama has access to the GPU resources.

Now, let's start Ollama in a separate thread so we can use it throughout this notebook.

We then start ollama in a separate thread so that we can use it after

Reference: https://stackoverflow.com/a/77828874

In [2]:
import os
import asyncio
import threading

# NB: You may need to set these depending and get cuda working depending which backend you are running.
# Set environment variable for NVIDIA library
# Set environment variables for CUDA
os.environ['PATH'] += ':/usr/local/cuda/bin'
# Set LD_LIBRARY_PATH to include both /usr/lib64-nvidia and CUDA lib directories
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64'

async def run_process(cmd, stdout=None, stderr=None):
    print('>>> starting', *cmd)
    process = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=stdout or asyncio.subprocess.PIPE,
        stderr=stderr or asyncio.subprocess.PIPE
    )

    if stdout is None and stderr is None:
        async def pipe(lines):
            async for line in lines:
                print(line.decode().strip())

        await asyncio.gather(
            pipe(process.stdout),
            pipe(process.stderr),
        )
    else:
        await process.wait()

async def start_ollama_serve():
    await run_process(['ollama', 'serve'],
                      stdout=open(os.devnull, 'w'),
                      stderr=open(os.devnull, 'w'))

def run_async_in_thread(loop, coro):
    asyncio.set_event_loop(loop)
    loop.run_until_complete(coro)
    loop.close()

# Create a new event loop that will run in a new thread
new_loop = asyncio.new_event_loop()

# Start ollama serve in a separate thread so the cell won't block execution
thread = threading.Thread(target=run_async_in_thread, args=(new_loop, start_ollama_serve()))
thread.start()

# Wait 5s for ollama to load
import time
time.sleep(5)

>>> starting ollama serve


## Playing with ollama API

In [3]:
# We first need to install dependencies
!pip install openai pydantic instructor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.5/328.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.6/327.6 kB 22.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 9.7 MB/s eta 0:00:00


For this first example, we will use [gemma2](https://blog.google/technology/developers/google-gemma-2/) from Google in its 9 billion parameters version. Its currently [one of the best small](https://chat.lmsys.org/?leaderboard) (<12B parameters) and opensource model.


In [4]:
!ollama pull gemma2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling ff1d1fc78170...   0% ▕▏    0 B/5.4 GB                  pulling manifest 
pulling ff1d1fc78170...   0% ▕▏    0 B/5.4 GB                  pulling manifest 
pulling ff1d1fc78170...   0% ▕▏    0 B/5.4 GB                  pulling manifest 
pulling ff1d1fc78170...   0% ▕▏    0 B/5.4 GB                  pulling manifest 
pulling ff1d1fc78170...   0% ▕▏    0 B/5.4 GB                  pulling manifest 
pulling ff1d1fc78170...   0% ▕▏    0 B/5.4 GB

In [5]:
MODEL = "gemma2"

## Use Case I: Generating malware information cards

In cybersecurity, having quick access to malware information is crucial for threat analysis and incident response.
We'll use our LLM to generate concise "malware cards" based on malware names. This demonstrates how LLMs can be
used to quickly summarize and present complex security information.

However, it's important to note:
1. This information is generated based on the LLM's training data and may not always be up-to-date or accurate.
2. **Always verify critical information with authoritative sources before making security decisions**.
3. Be cautious about potential hallucinations, especially for lesser-known malware.

In [6]:
!pip install openai pydantic instructor jsonref

#### Fetch malware data

In [7]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime

import instructor


class Malware(BaseModel):
    name: str
    first_seen: datetime
    language: str
    architecture: str
    developer: str
    description: str

def get_malware_description(malware_name: str) -> Malware:
  client = instructor.from_openai(
      OpenAI(
          base_url="http://127.0.0.1:11434/v1",
          api_key="ollama",  # required, but unused
      ),
      mode=instructor.Mode.JSON,
  )

  resp = client.chat.completions.create(
      model=MODEL,
      messages=[
          {
              "role": "user",
              "content": malware_name,
          }
      ],
      response_model=Malware,
  )
  return resp

malwares = ["Wannacry", "Stuxnet", "Darkcomet"]
for malware in malwares:
  resp = get_malware_description(malware)
  print(resp.model_dump_json(indent=2))

{
  "name": "WannaCry",
  "first_seen": "2017-05-12T00:00:00Z",
  "language": "Assembly",
  "architecture": "x86, x64",
  "developer": "Lazarus Group (believed)",
  "description": "A highly contagious ransomware worm that spread rapidly across the globe in May 2017. It exploited vulnerabilities in Microsoft Windows systems to encrypt user files and demand ransom for their release."
}
{
  "name": "Stuxnet",
  "first_seen": "2010-06-27T00:00:00Z",
  "language": "English (possibly compiled from other languages)",
  "architecture": "Windows x86, potentially others",
  "developer": "Unknown, highly suspected to be a joint US-Israeli effort",
  "description": "A sophisticated computer worm designed to sabotage Iranian nuclear facilities. It targeted and manipulated industrial control systems (ICS) used in uranium enrichment centrifuges."
}
{
  "name": "DarkComet",
  "first_seen": "2007-01-01T00:00:00Z",
  "language": "C/C++",
  "architecture": "32-bit, 64-bit",
  "developer": "Unknown",
  "d

## Use Case II: Building a daily cybersecurity news digest

Staying informed about the latest cybersecurity developments is essential but time-consuming. This use case
demonstrates how to leverage LLMs to create a personalized, AI-curated news digest.

Benefits:
1. Time-saving: Quickly get an overview of the most important cybersecurity news.
2. Prioritization: Focus on high-impact stories first.
3. Contextual understanding: Get AI-generated summaries and potential implications of each story.

The process involves:
1. Fetching articles from trusted RSS feeds
2. Using the LLM to rank and summarize the articles
3. Generating a comprehensive digest with actionable insights

Let's walk through each step:

In [8]:
!pip install feedparser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6049 sha256=a1d97108fcffe740ef8e16ccb6f54fba1bb77c88d03c273b32afbdf5d87771ad
  Stored in directory: /root/.cache/pip/wheels/f0/69/93/a47e9d621be168e9e33c7ce60524393c0b92ae83cf6c6e89c5
Successfully built sgmllib3k


#### Fetching of RSS entries

In [9]:
from pydantic import BaseModel, Field
from datetime import datetime
import uuid

class Article(BaseModel):
    id: str = Field(..., description="Unique identifier for the article")
    title: str = Field(..., description="Title of the article")
    link: str = Field(..., description="URL of the article")
    published: datetime = Field(..., description="Date and time the article was published")

import feedparser
import uuid

RSS_FEEDS = [
    "https://feeds.feedburner.com/TheHackersNews",
    "https://www.bleepingcomputer.com/feed/",
]

# fetch last rss entries and return title, url and id
def fetch_rss_entries(url):
    feed = feedparser.parse(url)
    return [Article(id=uuid.uuid4().hex[:8],
                   title=entry.title,
                   link=entry.link,
                   published=datetime(*entry.published_parsed[:6]))
            for entry in feed.entries]

articles = []
for rss_feed in RSS_FEEDS:
  articles.extend(fetch_rss_entries(rss_feed))
articles

[Article(id='40133ecc', title="'Konfety' Ad Fraud Uses 250+ Google Play Decoy Apps to Hide Malicious Twins", link='https://thehackernews.com/2024/07/konfety-ad-fraud-uses-250-google-play.html', published=datetime.datetime(2024, 7, 16, 13, 0)),
 Article(id='2a6192a5', title='Threat Prevention & Detection in SaaS Environments - 101', link='https://thehackernews.com/2024/07/threat-prevention-detection-in-saas.html', published=datetime.datetime(2024, 7, 16, 11, 0)),
 Article(id='12cd715d', title='Malicious npm Packages Found Using Image Files to Hide Backdoor Code', link='https://thehackernews.com/2024/07/malicious-npm-packages-found-using.html', published=datetime.datetime(2024, 7, 16, 10, 9)),
 Article(id='e527f513', title='Iranian Hackers Deploy New BugSleep Backdoor in Middle East Cyber Attacks', link='https://thehackernews.com/2024/07/iranian-hackers-deploy-new-bugsleep.html', published=datetime.datetime(2024, 7, 16, 9, 13)),
 Article(id='168ce78a', title='Void Banshee APT Exploits Mi

#### Ranking using LLM

In [20]:
from pydantic import BaseModel, Field
from typing import List
import instructor
from openai import OpenAI

class ArticleRef(BaseModel):
    id: str
    relevance_score: int = Field(..., ge=1, le=10, description="Relevance score from 1-10")

class RankedCyberNews(BaseModel):
    title: str = Field(..., max_length=100, description="Concise title of the cybersecurity news")
    summary: str = Field(..., max_length=250, description="Brief summary of the news")
    impact_score: int = Field(..., ge=1, le=10, description="Impact score from 1-10")
    related_articles: List[ArticleRef] = Field(..., min_items=1, description="List of related article IDs with relevance scores")

def rank_cyber_news(articles: List[Article]) -> List[RankedCyberNews]:
    client = instructor.from_openai(
        OpenAI(
            base_url="http://127.0.0.1:11434/v1",
            api_key="ollama",  # required, but unused
        ),
        mode=instructor.Mode.JSON,
    )

    article_data = "\n".join([f"{a.id} - {a.title}" for a in articles])
    prompt = f"""As a senior cybersecurity analyst, analyze the following articles and identify the 3 most significant cybersecurity news items. For each news item:

1. Provide a concise title (max 100 characters).
2. Write a brief summary (max 250 characters).
3. Assign an impact score (1-10) based on its potential consequences for individuals, organizations, or global cybersecurity.
4. List related article IDs with a relevance score (1-10) for each.

Consider factors such as:
- The scale and severity of any breaches or vulnerabilities
- The prominence of affected organizations or systems
- The novelty or sophistication of attack methods
- Potential long-term implications for cybersecurity practices

Articles to analyze:
{article_data}

Provide your analysis in a structured format suitable for JSON parsing.
"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_model=List[RankedCyberNews],
    )
    return resp

ranked_news = rank_cyber_news(articles)
ranked_news

[RankedCyberNews(title='Infostealer Garden: Low-Hanging Fruit for Hackers', summary="Over 10,000 victims infected daily by 'Infostealer Garden', a malware exploiting easily exploitable weaknesses.", impact_score=8, related_articles=[ArticleRef(id='ccde8788', relevance_score=10)]),
 RankedCyberNews(title='Konfety Ad Fraud: Decoy Apps Hide Malicious Twins', summary="'Konfety' campaign uses over 250 decoy Google Play apps to mask malicious twins, stealing ad revenue and user data.", impact_score=7, related_articles=[ArticleRef(id='40133ecc', relevance_score=10)]),
 RankedCyberNews(title='Malicious npm Packages Found with Backdoor Code', summary='Backdoor code hidden within image files in malicious npm packages, potentially compromising developer projects and software supply chains.', impact_score=6, related_articles=[ArticleRef(id='12cd715d', relevance_score=10)])]

In [11]:
# For each subject, we will parse articles using newspaper lib and ask for an actionnable summary
!pip install newspaper4k

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.6/296.6 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 15.8 MB/s eta 0:00:00


#### Article scraping

In [21]:
from newspaper import Article as ArticleParser

full_text = ""

for ranked_article in ranked_news:
    full_text += f"# News: {ranked_article.title}\n\n"
    full_text += f"Summary: {ranked_article.summary}\n\n"
    full_text += f"Impact Score: {ranked_article.impact_score}\n\n"

    # Select the article with the highest relevance score
    if ranked_article.related_articles:
        most_relevant_article = max(ranked_article.related_articles, key=lambda x: x.relevance_score)

        # Get article link using id
        article_link = next((a.link for a in articles if a.id == most_relevant_article.id), None)
        if article_link:
            full_text += f"# Most Relevant Article (Relevance: {most_relevant_article.relevance_score}):\n\n{article_link}\n\n"
            try:
                article_obj = ArticleParser(article_link)
                article_obj.download()
                article_obj.parse()
                full_text += f"# Text:\n\n{article_obj.text}\n\n"  # Limit to first 1000 characters
            except Exception as e:
                full_text += f"Error fetching article: {str(e)}\n\n"
    else:
        full_text += "No related articles found for this news item.\n\n"

print(f"Processed {len(ranked_news)} news items.")

Processed 3 news items.


#### Digest building

In [22]:
from openai import OpenAI

def summarize_articles(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    As an expert cybersecurity analyst, create a concise yet comprehensive daily news digest in markdown format. Your task:

    1. Analyze the provided cybersecurity articles.
    2. Summarize the given stories, which are already ranked by importance.
    3. For each story, include:
       - The provided headline
       - A brief summary (2-3 sentences) highlighting key points
       - The potential impact on individuals or organizations (consider the given impact score)
       - Key details from the most relevant related article
       - Actionable advice or takeaways for readers

    4. Use bullet points for clarity and readability.
    5. Conclude with a "Bottom Line" that provides an overall assessment of the day's cybersecurity landscape.

    Guidelines:
    - Use clear, professional language avoiding unnecessary jargon.
    - Prioritize stories based on their impact scores.
    - Ensure the total digest length is between 500-750 words.
    - Use markdown formatting to enhance readability (e.g., headers, bold for emphasis).

    Articles to summarize:

    {full_text}

    Produce only the markdown-formatted digest without any additional commentary or metadata.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return resp.choices[0].message.content

digest = summarize_articles(full_text)
print(digest)

## News: Malicious npm Packages Found with Backdoor Code

Cybersecurity researchers have identified two malicious packages on the npm package registry that concealed backdoor code to execute malicious commands sent from a remote server. The packages, img-aws-s3-object-multipart-copy and legacyaws-s3-object-multipart-copy, have been downloaded 190 and 48 times each. They have been taken down by the npm security team.

The packages are designed to impersonate a legitimate npm library called aws-s3-object-multipart-copy, but come with an altered version of the "index.js" file to execute a JavaScript file ("loadformat.js"). This file processes three images – featuring logos for Intel, Microsoft, and AMD – using the image corresponding to Microsoft's logo to extract and execute malicious content.

The code registers the new client with a command-and-control (C2) server by sending hostname and operating system details. It then attempts to execute attacker-issued commands periodically every f

#### Bonus: build your daily podcast

In this part, we will use another prompt to generate the script for a podcast about the last cybersecurity news. We will then use

In [14]:
def build_podcast(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    You are an engaging cybersecurity podcast host. Create a script for a 5-minute daily news digest based on the provided articles. Your task:

    1. Start with a brief, catchy introduction.
    2. Summarize 2-3 key cybersecurity stories from the given content.
    3. For each story, provide:
       - A concise overview of the incident or development
       - Its potential impact on individuals or organizations
       - Quick, actionable advice for listeners (if applicable)
    4. Use a conversational tone, as if speaking directly to the audience.
    5. Conclude with a brief sign-off, encouraging listeners to stay vigilant.

    Guidelines:
    - Keep the total length around 600-800 words.
    - Use simple language and avoid technical jargon when possible.
    - Include relevant statistics or numbers to add context.
    - Add brief transitions between stories to maintain flow.

    Content to summarize:

    {full_text}

    Remember, create only the podcast script without any formatting characters or annotations.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return resp.choices[0].message.content

podcast = build_podcast(full_text)
podcast

"Welcome back to edList.io's cyber security news roundup, where we break down the latest threats and how to stay safe. Today's top story is about a sophisticated threat group known as Void Banshee exploiting a newly discovered Microsoft flaw. \n\nCybersecurity researchers at Trend Micro have uncovered an attack campaign where Void Banshee uses a vulnerability in the Microsoft MHTML browser engine to deliver Atlantida, a powerful information stealer.  This vulnerability, tracked as CVE-2024-38112, was actively used by Void Banshee starting in mid-May 2024. What makes this particularly concerning is that Void Banshee exploited the vulnerability through disabled services like Internet Explorer, highlighting how even seemingly outdated software can pose a risk.\n\nThe attack chain involves spear phishing emails with links to specially crafted URL files. These files, when opened, exploit the Microsoft MHTML flaw and redirect victims to malicious websites hosting an infected HTML application

In [15]:
!git clone https://github.com/myshell-ai/MeloTTS.git
%cd MeloTTS
!pip install -e .
!python -m unidic download

Cloning into 'MeloTTS'...
remote: Enumerating objects: 417, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 417 (delta 165), reused 133 (delta 133), pack-reused 212
Receiving objects: 100% (417/417), 6.04 MiB | 3.37 MiB/s, done.
Resolving deltas: 100% (212/212), done.
/content/MeloTTS
Obtaining file:///content/MeloTTS
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 76.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.1/581.1 kB 45.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:
from melo.api import TTS
# Speed is adjustable
speed = 1.1

# CPU is sufficient for real-time inference.
# You can set it manually to 'cpu' or 'cuda' or 'cuda:0' or 'mps'
device = 'auto' # Will automatically use GPU if available

# English
model = TTS(language='EN', device=device)
speaker_ids = model.hps.data.spk2id

# Default accent
output_path = 'en-default.wav'
model.tts_to_file(podcast, speaker_ids['EN-Default'], output_path, speed=speed)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

checkpoint.pth:   0%|          | 0.00/208M [00:00<?, ?B/s]

 > Text split to sentences.
Welcome back to edList. io's cyber security news roundup, where we break down the latest threats and how to stay safe. Today's top story is about a sophisticated threat group known as Void Banshee exploiting a newly discovered Microsoft flaw. Cybersecurity researchers at Trend Micro have uncovered an attack campaign where Void Banshee uses a vulnerability in the Microsoft MHTML browser engine to deliver Atlantida,
a powerful information stealer. This vulnerability, tracked as CVE-2024-38112, was actively used by Void Banshee starting in mid-May 2024. What makes this particularly concerning is that Void Banshee exploited the vulnerability through disabled services like Internet Explorer,
highlighting how even seemingly outdated software can pose a risk. The attack chain involves spear phishing emails with links to specially crafted URL files. These files, when opened, exploit the Microsoft MHTML flaw and redirect victims to malicious websites hosting an infec

  0%|          | 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/conv.py:797: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv_transpose1d(
/usr/local/lib/python3.10/dist-pa

In [18]:
from IPython.display import Audio #Import Audio method from IPython's Display Class

Audio(output_path, autoplay=False)

Output hidden; open in https://colab.research.google.com to view.

## Conclusion

In this notebook, we've explored how to leverage Large Language Models (LLMs) for cybersecurity applications using Ollama. We've demonstrated two practical use cases:

1. Generating malware information cards for quick reference and analysis.
2. Creating an AI-curated cybersecurity news digest to stay informed about the latest threats and developments.

These examples showcase the potential of LLMs to enhance cybersecurity operations by:
- Quickly synthesizing and presenting complex information
- Automating time-consuming tasks like news curation and summarization
- Providing insights that can aid in threat analysis and decision-making

However, it's crucial to remember that while LLMs are powerful tools, they should be used in conjunction with human expertise and established cybersecurity practices. **Always verify AI-generated information, especially when making critical security decisions**.

Stay curious, stay secure, and keep pushing the boundaries of what's possible in this new field :)